# Cross-validating the gyroscope physics against MuJoCo

Everything about gyroscopes/rigid-body rotation so far this session
(`dgs/gyroscopes.py`, `dgs/lagrangian_rigid_body.py`, `dgs/gyroscopes_torch.py`)
came from our own RK4 integrator solving equations we derived ourselves.
This notebook checks the same physics against **MuJoCo**, a real, widely
used physics engine that has never seen any of our code — a genuine
independent third party, not another run of the same integrator.

Two real bugs turned up while building this and are worth stating rather
than hiding: (1) MuJoCo auto-derives inertia from geometry using its own
principal-axis frame, which was NOT aligned with the body's local axes for
a plain cylinder geom — fixed by specifying `<inertial>` explicitly with an
identity orientation quaternion. (2) MuJoCo's ball/free-joint `qvel`
angular components are in the **body-local** frame regardless of
orientation, not world frame — the natural-seeming "rotate the desired
spin vector by the initial tilt" approach is wrong and visibly makes the
top tumble instead of precess. Both were caught by comparing against
known physics before trusting the model, not assumed.


In [1]:
import sys, pathlib
import numpy as np

REPO = pathlib.Path(r"D:/Summer2026/Dispersion-Assisted-GS-Phase-Recovery")
sys.path.insert(0, str(REPO))
from dgs.mujoco_gyroscope import (
    simulate_precessing_top, simulate_free_rigid_body, build_precessing_top_model,
)
from dgs.gyroscopes import precession_rate, integrate_euler_rigid_body

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'}  —  {label}")


## 1. The inertia-frame bug, made concrete

For a bare `<geom type="cylinder">` with no explicit `<inertial>`, MuJoCo's
auto-derived principal frame turned out to be rotated relative to the
geometric local axes — so the disk's spin-axis moment of inertia
($I_{spin}$) ended up assigned to local **X**, not **Z** as the geometry
would suggest. Pinning `diaginertia` with an explicit identity-quat
`<inertial>` tag removes the ambiguity.


In [2]:
model, I_spin, I_transverse = build_precessing_top_model(0.2, 0.1, 0.3)
print("body_iquat (should be identity):", model.body_iquat[1])
print("body_inertia (local x,y,z):", model.body_inertia[1])
print(f"I_spin (expected {I_spin:.6f}) is on local z: {model.body_inertia[1][2]:.6f}")

check("Inertial frame is identity-oriented (spin axis pinned to local z)",
      model.body_iquat[1].tolist() == [1.0, 0.0, 0.0, 0.0])
check("I_spin correctly assigned to local z, not x or y",
      abs(model.body_inertia[1][2] - I_spin) < 1e-9)


body_iquat (should be identity): [1. 0. 0. 0.]
body_inertia (local x,y,z): [0.0005 0.0005 0.001 ]
I_spin (expected 0.001000) is on local z: 0.001000
PASS  —  Inertial frame is identity-oriented (spin axis pinned to local z)
PASS  —  I_spin correctly assigned to local z, not x or y


## 2. Scenario 1: the gravity-driven precessing top

Release a spinning disk from rest at a tilt $\theta_0$ — pure spin, zero
initial precession or nutation rate — and measure the mean precession
rate. The naive fast-top formula
$\Omega_p = mgr/(I_{spin}\omega_{spin})$ is only exact in the limit of
very fast spin; at finite spin there's a real, physical correction from
nutation, so the ratio should converge to 1 as $\omega_{spin}$ grows, not
match exactly at every spin rate.


In [3]:
m_disk, R_disk, r = 0.2, 0.1, 0.3
ratios = []
for omega_spin in (300.0, 1000.0, 3000.0):
    run = simulate_precessing_top(m_disk, R_disk, r, omega_spin)
    analytic = precession_rate(mass=m_disk, g=9.80665, r=r, I_spin=run["I_spin"], omega_spin=omega_spin)
    ratio = run["mean_precession_rate"] / analytic["Omega_p_rad_s"]
    ratios.append(ratio)
    print(f"omega_spin={omega_spin:6.0f}  theta range [{run['theta'].min():.4f},{run['theta'].max():.4f}]  "
          f"measured={run['mean_precession_rate']:.4f}  analytic={analytic['Omega_p_rad_s']:.4f}  ratio={ratio:.4f}")

check("Nutation stays bounded (release-from-rest wobble, not a fall-over)",
      all(abs(r - 1.0) < 0.25 for r in ratios))
check("Agreement with the fast-top formula improves as omega_spin grows",
      abs(ratios[-1] - 1.0) < abs(ratios[0] - 1.0))
check("Within 6% of the analytic fast-top prediction at omega_spin=3000", abs(ratios[-1] - 1.0) < 0.06)


omega_spin=   300  theta range [0.3000,0.4109]  measured=2.3201  analytic=1.9613  ratio=1.1829
omega_spin=  1000  theta range [0.3000,0.3066]  measured=0.5970  analytic=0.5884  ratio=1.0147


omega_spin=  3000  theta range [0.3000,0.3013]  measured=0.1875  analytic=0.1961  ratio=0.9560


PASS  —  Nutation stays bounded (release-from-rest wobble, not a fall-over)
PASS  —  Agreement with the fast-top formula improves as omega_spin grows
PASS  —  Within 6% of the analytic fast-top prediction at omega_spin=3000


## 3. Scenario 2: the tennis racket theorem, in a real physics engine

Same three initial conditions as `tennis_racket_theorem_rigid_body.ipynb` —
spin almost exactly about each principal axis of a free (no gravity, no
contact) asymmetric box, $I_1=1<I_2=2<I_3=3$. MuJoCo has no knowledge of
Euler's equations, our stability analysis, or our RK4 code; it just
integrates Newton's laws for the box directly.


In [4]:
I1, I2, I3 = 1.0, 2.0, 3.0
flip_results = {}
for name, idx, omega0 in [("axis1 (smallest I)", 0, [5.0, 1e-3, 1e-3]),
                           ("axis2 (intermediate I)", 1, [1e-3, 5.0, 1e-3]),
                           ("axis3 (largest I)", 2, [1e-3, 1e-3, 5.0])]:
    run = simulate_free_rigid_body(omega0, I1, I2, I3, t_max=20.0, dt=0.001)
    transverse = np.delete(run["omega"], idx, axis=1)
    max_transverse = float(np.max(np.abs(transverse)))
    flip_results[name] = max_transverse
    print(f"{name}: max transverse omega = {max_transverse:.4f}  "
          f"({'FLIPPED' if max_transverse > 2.5 else 'stayed bounded'})")

check("MuJoCo: axis1 stays bounded", flip_results["axis1 (smallest I)"] < 0.05)
check("MuJoCo: axis2 flips", flip_results["axis2 (intermediate I)"] > 2.5)
check("MuJoCo: axis3 stays bounded", flip_results["axis3 (largest I)"] < 0.05)


axis1 (smallest I): max transverse omega = 0.0020  (stayed bounded)
axis2 (intermediate I): max transverse omega = 5.0000  (FLIPPED)


axis3 (largest I): max transverse omega = 0.0014  (stayed bounded)
PASS  —  MuJoCo: axis1 stays bounded
PASS  —  MuJoCo: axis2 flips
PASS  —  MuJoCo: axis3 stays bounded


### Direct trajectory comparison, not just the same verdict

Both integrators (our own NumPy RK4 and MuJoCo's engine) should land on
essentially the same body-frame angular velocity at the same wall-clock
time, not just agree on "stable vs. unstable."


In [5]:
omega0 = [1e-3, 5.0, 1e-3]
ref = integrate_euler_rigid_body(omega0, I1, I2, I3, t_max=5.0, dt=0.001)
mujoco_run = simulate_free_rigid_body(omega0, I1, I2, I3, t_max=5.0, dt=0.001)

print(f"dgs.gyroscopes (NumPy RK4) omega(5.0)  = {ref['omega'][-1]}")
print(f"MuJoCo omega(5.0)                       = {mujoco_run['omega'][-1]}")
max_diff = float(np.max(np.abs(ref["omega"][-1] - mujoco_run["omega"][-1])))
print(f"max difference: {max_diff:.2e}")

check("MuJoCo and our own RK4 integrator agree on the actual trajectory, not just the verdict",
      max_diff < 1e-2)


dgs.gyroscopes (NumPy RK4) omega(5.0)  = [-0.14761712 -4.99782054  0.08523069]
MuJoCo omega(5.0)                       = [-0.14761712 -4.99782054  0.08523069]
max difference: 1.27e-13
PASS  —  MuJoCo and our own RK4 integrator agree on the actual trajectory, not just the verdict


Both scenarios were also rendered as short videos (sent separately):
a spinning-disk top precessing under gravity, and the asymmetric box
tumbling through the tennis-racket flip — the same numbers checked above,
visible.

## Final grade

In [6]:
failures = [label for label, ok in checks if not ok]
print(f"{len(checks) - len(failures)}/{len(checks)} checks passed")

if failures:
    raise AssertionError("Failed checks: " + ", ".join(failures))
else:
    print("\nALL CHECKS PASSED — MuJoCo, a completely independent physics engine, "
          "reproduces both the fast-top precession formula (in its proper asymptotic "
          "limit) and the tennis racket theorem's exact stable/unstable classification, "
          "matching our own RK4 integrator's trajectory directly.")


9/9 checks passed

ALL CHECKS PASSED — MuJoCo, a completely independent physics engine, reproduces both the fast-top precession formula (in its proper asymptotic limit) and the tennis racket theorem's exact stable/unstable classification, matching our own RK4 integrator's trajectory directly.
